# Day 11 — Explainable AI

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", context="notebook")

from src.explainability import (
    explain_with_shap,
    permutation_importance_table,
    plot_importance_bar,
    save_explainability_figure,
    top_feature_importances,
)
from src.inference import load_production_classifier
from src.preprocessing import load_cleaned_table, prepare_train_test_bundle, load_artifacts
from src.tree_models import build_random_forest


In [ ]:
artifacts = load_artifacts(PROJECT_ROOT)
bundle = prepare_train_test_bundle(load_cleaned_table(PROJECT_ROOT))
feature_names = artifacts["feature_names"]
model, _, model_name = load_production_classifier(PROJECT_ROOT)

if hasattr(model, "feature_importances_"):
    imp = top_feature_importances(model, feature_names)
else:
    rf = build_random_forest()
    rf.fit(bundle["X_train"], bundle["y_train"])
    imp = top_feature_importances(rf, feature_names)

fig = plot_importance_bar(imp, title=f"Importances ({model_name})")
save_explainability_figure(fig, PROJECT_ROOT, "day11_feature_importance.png")
imp

In [ ]:
shap_df = explain_with_shap(
    model if hasattr(model, "feature_importances_") else rf,
    bundle["X_train"],
    bundle["X_test"][:1],
    feature_names,
)
shap_df

Use careful wording: features **contributed to the model score**, not destiny.